# Rosette MRSI pipeline (pure-Python FID-A port)

Mirrors `run_MRSI_Rosette_40x40.m` step-by-step. The `mrsi/` folder is added to
`sys.path` (the `addpath(genpath(...))` equivalent) so every `op_*.py` function is
importable by name, then called one at a time. Outputs land in the dataset's
`outputs/` folder, the same place FID-A writes.

**Config knobs** (below): recon `DCF`/`FT`, `SMOOTH_FWHM`, `PHANTOM` (skip SSP),
`ZEROFILL` (None = NUNFIL 576 = matches the FID-A subject02 RAW; 4096 to interpolate).

In [1]:
# --- setup: put the entire mrsi/ folder on the path, import each function ---
import os, sys
import numpy as np

ROOT     = r'C:\Users\divya\Downloads\mrsi_pipeline'
MRSI_DIR = os.path.join(ROOT, 'mrsi')
sys.path.insert(0, MRSI_DIR)   # addpath(genpath(mrsi))  -> every op_*.py importable
sys.path.insert(0, ROOT)       # for fit_lcmodel_rosette + viz

from io_CSIload_twix     import io_CSIload_twix_pair
from op_CSIRosettePrep   import prep_noncartesian
from op_CSIRecon         import op_CSIRecon
from op_CSICombineCoils1 import op_CSICombineCoils1
from op_CSIAverage       import op_CSIAverage
from op_CSISegment_simple import op_CSISegment_simple
from op_CSIleftshift     import op_CSIleftshift
from op_CSIFourierTransform import op_CSIFourierTransform
from op_CSIssp           import op_CSIssp
from op_CSIRemoveLipids  import op_CSIRemoveLipids
from op_CSIB0Correction_v2 import op_CSIB0Correction_v2
from op_CSIspecZeroFill  import op_CSIspecZeroFill
from op_CSIapplymask     import op_CSIapplymask
from op_CSIApodize       import op_CSIApodize
from op_CSIFlip180       import op_CSIFlip180
from fit_lcmodel_rosette import fit_maps
from viz._common         import to_fyx
print('imported', len([f for f in dir() if f.startswith('op_')]), 'op_ functions')

imported 13 op_ functions


In [2]:
# --- USER INPUTS (match run_MRSI_Rosette_40x40.m) ---
DATASET = r'F:\fida\divya\20260605_phantom_test\subject02'
MET   = os.path.join(DATASET, 'met',     'meas_MID00138_FID48082_Rosette_40x40_isoctr.dat')
REF   = os.path.join(DATASET, 'mrs_ref', 'meas_MID00139_FID48083_Rosette_40x40_isoctr_w.dat')
KFILE = r'C:\Users\divya\Downloads\fida codes\fid_a\processingTools\MRSI\kFiles\Rosette_traj_40x40.txt'

# output stored in the SAME dataset location as FID-A (outputs/); _py suffix keeps
# FID-A's own lcm_out intact -- drop the suffix to write into FID-A's exact folder.
OUT = os.path.join(DATASET, 'outputs', 'lcm_out_py')
os.makedirs(OUT, exist_ok=True)

# reconstruction + processing choices
DCF, FT      = 'pipe_menon', 'nufft'   # FID-A production config
SMOOTH_FWHM  = 20                       # spatial Gaussian FWHM (mm)
PHANTOM      = True                     # phantom -> SKIP SSP (0.8-1.88 band removes Lac)
ZEROFILL     = None                     # None = NUNFIL 576 (matches subject02 RAW); 4096 to interpolate
DO_LCMODEL   = True                     # set False to stop after apodize (fast)
print('dataset:', DATASET, '\noutput ->', OUT)

dataset: F:\fida\divya\20260605_phantom_test\subject02 
output -> F:\fida\divya\20260605_phantom_test\subject02\outputs\lcm_out_py


## 1. Load TWIX pair  (`io_CSIload_twix_pair`)

In [3]:
tc, tc_w = io_CSIload_twix_pair(MET, REF, KFILE, 'rosette')
print('met', tc['data'].shape, tc['dims'])

read data:   0%|          | 0/1134 [00:00<?, ?it/s]

read data:   0%|          | 0/567 [00:00<?, ?it/s]

met (8064, 16, 63, 9, 2) {'t': 1, 'coils': 2, 'averages': 5, 'timeinterleave': 0, 'kx': 0, 'ky': 0, 'kz': 0, 'x': 0, 'y': 0, 'z': 0, 'kpts': 0, 'kshot': 3, 'subspec': 0, 'extras': 4, 'f': 0}


## 1b. Rosette prep  (`prep_noncartesian`)

Reshape the raw readout into `[t, coils, avg, kpts, kshot]` (adds the kpts/kshot
trajectory dims recon needs). FID-A's loader returns this already-reshaped; the
Python port keeps it as an explicit step.

In [4]:
tc   = prep_noncartesian(tc,   KFILE, 'rosette')
tc_w = prep_noncartesian(tc_w, KFILE, 'rosette')
print('prepped met', tc['data'].shape, tc['dims'])

prepped met (576, 126, 16, 63, 2) {'t': 1, 'coils': 3, 'averages': 5, 'timeinterleave': 0, 'kx': 0, 'ky': 0, 'kz': 0, 'x': 0, 'y': 0, 'z': 0, 'kpts': 2, 'kshot': 4, 'subspec': 0, 'extras': 0, 'f': 0}


## 2. Spatial reconstruction  (`op_CSIRecon`)  — DCF + FT

In [5]:
ft   = op_CSIRecon(tc,   KFILE, DCF, FT)
ft_w = op_CSIRecon(tc_w, KFILE, DCF, FT)
print('recon met', ft['data'].shape)

recon met (576, 16, 2, 40, 40)


## 3. Coil combination  (`op_CSICombineCoils1`) — Roemer, maps from water ref

In [6]:
cc_w, phase, weights = op_CSICombineCoils1(ft_w)
cc                    = op_CSICombineCoils1(ft, 1, phase, weights)[0]
print('combined met', cc['data'].shape)

combined met (576, 2, 40, 40)


## 4. Average + water mask  (`op_CSIAverage`, `op_CSISegment_simple`)

In [7]:
ccav   = op_CSIAverage(cc)
ccav_w = op_CSIAverage(cc_w)
ccav_w = op_CSISegment_simple(ccav_w)
ccav['mask'] = ccav_w['mask']
mask = ccav_w['mask']['brainmasks']

op_CSISegment_simple: 507/1600 voxels (31.7%)  threshold=3.64e-05


## 4b. Left-shift  (`op_CSIleftshift`) — remove FID first-point phase (ls=0 here -> no-op)

In [8]:
ccav   = op_CSIleftshift(ccav)
ccav_w = op_CSIleftshift(ccav_w)

## 5. Spectral FT  (`op_CSIFourierTransform`, spatial already done in recon)

In [9]:
ftSpec   = op_CSIFourierTransform(ccav,   spatial=False, spectral=True)
ftSpec_w = op_CSIFourierTransform(ccav_w, spatial=False, spectral=True)
print('ppm', ftSpec['ppm'][0], '..', ftSpec['ppm'][-1])

ppm 11.089210961311807 .. -1.7668525899183614


## 6. Lipid/water removal + B0  (`op_CSIssp` skipped for phantom, `op_CSIRemoveLipids`, `op_CSIB0Correction_v2`)

In [10]:
if PHANTOM:
    rmlip = ftSpec                          # skip SSP: 0.8-1.88 ppm band removes lactate
else:
    rmlip = op_CSIssp(ftSpec, 0.8, 1.88)
# water removal in the [4.5 5.0] ppm band (FID-A run_MRSI_Rosette_40x40 settings)
ftSpec_rmw = op_CSIRemoveLipids(rmlip, lipidPPMRange=(4.5, 5.0), lineWidthRange=(1, 10))
ftSpec_B0, ftSpec_B0_w, freqMap, R2Map = op_CSIB0Correction_v2(ftSpec_rmw, ftSpec_w)

## 6b. (optional) spectral zero-fill  (`op_CSIspecZeroFill`)

In [11]:
if ZEROFILL:
    ftSpec_B0   = op_CSIspecZeroFill(ftSpec_B0,   ZEROFILL)
    ftSpec_B0_w = op_CSIspecZeroFill(ftSpec_B0_w, ZEROFILL)
    print('zero-filled ->', ftSpec_B0['data'].shape[0])
else:
    print('no zero-fill; NUNFIL =', ftSpec_B0['data'].shape[0])

no zero-fill; NUNFIL = 576


## 7. Apply mask + spatial smoothing  (`op_CSIapplymask`, `op_CSIApodize`)

In [12]:
ftSpec_B0['mask'] = ccav['mask']
ftSpec_masked = op_CSIapplymask(ftSpec_B0)
if SMOOTH_FWHM > 0:
    ftSpec_smooth   = op_CSIApodize(ftSpec_masked, 'gaussian', SMOOTH_FWHM)
    ftSpec_smooth_w = op_CSIApodize(ftSpec_B0_w,   'gaussian', SMOOTH_FWHM)
else:
    ftSpec_smooth, ftSpec_smooth_w = ftSpec_masked, ftSpec_B0_w
print('smoothed met', ftSpec_smooth['data'].shape)

smoothed met (576, 40, 40)


## 8. LCModel + metabolite maps  (`fit_maps`)

Per-voxel LCModel over the mask: writes RAW / control / table for every voxel into
`OUT` (FID-A layout) and returns conc / CRLB / LW / SNR maps + `maps.npz` + `lcm_maps.png`.

In [13]:
if DO_LCMODEL:
    met_fyx = to_fyx(ftSpec_smooth['data'],   ftSpec_smooth['dims'])
    wat_fyx = to_fyx(ftSpec_smooth_w['data'], ftSpec_smooth_w['dims'])
    res = fit_maps(met_fyx, wat_fyx, np.asarray(ftSpec_smooth['ppm']), mask, OUT)
    print('maps + tables ->', OUT)
else:
    print('DO_LCMODEL = False (skipped)')

LCModel: fitting 507 voxels ONE-BY-ONE (NUNFIL=576)...
  [  25/507]  25 fit OK
  [  50/507]  50 fit OK
  [  75/507]  75 fit OK
  [ 100/507]  100 fit OK
  [ 125/507]  125 fit OK
  [ 150/507]  150 fit OK
  [ 175/507]  175 fit OK
  [ 200/507]  200 fit OK
  [ 225/507]  225 fit OK
  [ 250/507]  250 fit OK
  [ 275/507]  275 fit OK
  [ 300/507]  300 fit OK
  [ 325/507]  325 fit OK
  [ 350/507]  350 fit OK
  [ 375/507]  375 fit OK
  [ 400/507]  400 fit OK
  [ 425/507]  425 fit OK
  [ 450/507]  450 fit OK
  [ 475/507]  475 fit OK
  [ 500/507]  500 fit OK
  [ 507/507]  507 fit OK
507/507 voxels fit OK
  Cr: median CRLB 2%  CRLB<=20 in 502 vox
  Cho: median CRLB 3%  CRLB<=20 in 502 vox
  Lac: median CRLB 999%  CRLB<=20 in 109 vox
  Act: median CRLB 3%  CRLB<=20 in 502 vox


C:\Users\divya\Downloads\mrsi_pipeline\fit_lcmodel_rosette.py:136: RuntimeWarning: divide by zero encountered in divide
  rat = conc['Cho'] / conc['Cr']; rat[(crlb['Cr'] > crlb_cap) | (crlb['Cho'] > crlb_cap) | ~mask] = np.nan


rendered F:\fida\divya\20260605_phantom_test\subject02\outputs\lcm_out_py\lcm_maps.png
maps + tables -> F:\fida\divya\20260605_phantom_test\subject02\outputs\lcm_out_py


## 9. View maps

In [14]:
np.shape(res['conc']['Cho'])
res['conc']['Cho'][1:2,1:3]

array([[nan, nan]])

In [15]:
OUT

'F:\\fida\\divya\\20260605_phantom_test\\subject02\\outputs\\lcm_out_py'

In [16]:
import matplotlib.pyplot as plt
from PIL import Image
img = os.path.join(OUT, 'lcm_maps.png')
if os.path.exists(img):
    plt.figure(figsize=(15, 10)); plt.imshow(Image.open(img)); plt.axis('off'); plt.show()
else:
    print('run step 8 first')

C:\Users\divya\AppData\Local\Temp\ipykernel_8744\1998259319.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(15, 10)); plt.imshow(Image.open(img)); plt.axis('off'); plt.show()
